# ViennaRNA Secondary Structure Prediction

![ViennaRNA Secondary Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/viennarna/hero.png)

This notebook demonstrates RNA secondary structure prediction using ViennaRNA. ViennaRNA uses thermodynamic nearest-neighbor models to compute the minimum free energy (MFE) fold for an RNA sequence, returning the predicted structure in dot-bracket notation along with the MFE in kcal/mol. Because it is a CPU-only tool with no model weights to load, predictions complete in milliseconds even for long sequences. The example folds five short RNA sequences that span a range of structural behaviors, from stable hairpins to unstructured stretches.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("viennarna")
display_overview("viennarna")
display_docs_section("viennarna", "Background")

# ViennaRNA

First released in 1994, ViennaRNA is a thermodynamic RNA secondary-structure prediction package. From an RNA or DNA sequence it computes the minimum-free-energy secondary structure and its free energy using a nearest-neighbor energy model, with no training and no GPU required. It is widely used to predict and compare the base-pairing of messenger RNAs, non-coding RNAs, riboswitches, and designed RNA constructs.

ViennaRNA ([Lorenz et al., 2011](https://doi.org/10.1186/1748-7188-6-26)) predicts the secondary structure of a nucleic-acid sequence: the set of intramolecular base pairs that form within a single strand. Secondary structure sits between sequence and three-dimensional shape and governs how many functional RNAs behave, so predicting it from sequence alone is a core step in RNA analysis and design.

Internally, ViennaRNA folds each sequence with the minimum-free-energy dynamic program at the core of RNAfold (cubic in the sequence length) under a nearest-neighbor thermodynamic model. RNA sequences use the Turner 2004 parameters. Selecting the DNA option instead loads the Mathews 2004 DNA parameters. The predicted structure is returned in dot-bracket notation together with its minimum free energy in kcal/mol, where a more negative value indicates a more stable predicted fold. Because it is a thermodynamic rather than a learned method, it is deterministic, runs on CPU, and predicts secondary structure only, not three-dimensional coordinates.

The reference implementation is the ViennaRNA Package, maintained by [TBI Vienna](https://www.tbi.univie.ac.at/) at [ViennaRNA/ViennaRNA](https://github.com/ViennaRNA/ViennaRNA).

## Available tools

In [2]:
display_available_tools("viennarna")

- **`run_viennarna()`** — RNA secondary structure prediction using ViennaRNA MFE folding

### `run_viennarna`

ViennaRNA computes the minimum free energy secondary structure of an RNA sequence using the thermodynamic nearest-neighbor model. The output structure uses dot-bracket notation where '.' denotes an unpaired base and matched '(' ')' pairs denote base pairs. More negative MFE values indicate more thermodynamically stable folds. The five sequences below span a range of structural behaviors: GC-rich stems with UUUU loops (highly stable), mixed stems, and unstructured AU-rich sequences (near-zero MFE), making them a useful comparison set for understanding how sequence composition drives fold stability.

In [3]:
from proto_tools import ViennaRNAInput, ViennaRNAConfig, run_viennarna

In [4]:
# Display input API reference
display_api_reference("viennarna", "input", "run_viennarna")

# A real tRNA plus designed hairpins, spanning a range of structural complexity.
inputs = ViennaRNAInput(sequences=[
    # Yeast tRNA-Phe (76 nt) folds into the classic cloverleaf secondary structure.
    "GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGGUCCUGUGUUCGAUCCACAGAAUUCGCACCA",
    "GCGCUUUUGCGC",  # GC-rich stem with a UUUU loop (stable hairpin)
    "GGGGAAACCCC",  # G-rich stem with an AAA loop
    "CGCGAAACGCG",  # alternating CG stem with an AAA loop
])

**Input** — `ViennaRNAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | List of input RNA sequences |

In [5]:
# Display config API reference
display_api_reference("viennarna", "config", "run_viennarna")

# Configure ViennaRNA folding at physiological temperature
config = ViennaRNAConfig(
    temperature=37.0,  # Physiological temperature in Celsius
)

**Config** — `ViennaRNAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>temperature</code> | <code>float</code> | <code>37.0</code> | Temperature in Celsius for energy calculations |
| <code>use_dna_params</code> | <code>bool</code> | <code>False</code> | Use DNA energy parameters instead of RNA parameters |
| <code>no_lonely_pairs</code> | <code>bool</code> | <code>False</code> | Disallow lonely base pairs (helices of length 1) |
| <code>dangles</code> | <code>Literal[0, 1, 2, 3]</code> | <code>2</code> | Dangling-end treatment: 0=ignore, 1=minimal, 2=multibranch, 3=accurate |
| <code>circ</code> | <code>bool</code> | <code>False</code> | Treat sequence as circular (plasmids, viroids, circRNAs) |
| <code>max_bp_span</code> | <code>int</code> | <code>-1</code> | Max base-pair span in nt; -1 = no limit, positive forbids long-range pairs |

In [6]:
# Run the prediction
result = run_viennarna(inputs, config)

Running run_viennarna [00:00]

In [7]:
# Display output API reference
display_api_reference("viennarna", "output", "run_viennarna")

# Display predicted structures and MFE for each sequence
for i, res in enumerate(result.results):
    print(f"Sequence {i + 1}: {res.sequence}")
    print(f"  Structure: {res.structure}")
    print(f"  MFE:       {res.mfe:.2f} kcal/mol")
    print()

**Output** — `ViennaRNAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[ViennaRNAResult]</code> | required | List of ViennaRNA results |

**`ViennaRNAResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | The input RNA sequence |
| <code>structure</code> | <code>str &#124; None</code> | <code>None</code> | Predicted RNA secondary structure in dot-bracket notation |
| <code>mfe</code> | <code>float &#124; None</code> | <code>None</code> | Minimum free energy in kcal/mol; more negative values indicate a more stable structure |

Sequence 1: GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGGUCCUGUGUUCGAUCCACAGAAUUCGCACCA
  Structure: (((((((..((((........)))).(((((.......))))).....(((((.......))))))))))))....
  MFE:       -22.40 kcal/mol

Sequence 2: GCGCUUUUGCGC
  Structure: ((((....))))
  MFE:       -5.70 kcal/mol

Sequence 3: GGGGAAACCCC
  Structure: ((((...))))
  MFE:       -4.50 kcal/mol

Sequence 4: CGCGAAACGCG
  Structure: ((((...))))
  MFE:       -2.80 kcal/mol



## Export Results

Folding results can be exported to CSV, JSON, or FASTA format for downstream analysis. The CSV format produces a simple table with sequence, structure, and MFE columns that can be loaded into a spreadsheet or any data-processing pipeline. The FASTA format includes the MFE as a sequence header annotation alongside the dot-bracket structure.

In [8]:
from pathlib import Path

# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to CSV
result.export("viennarna_results", export_path=output_dir, file_format="csv")